<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/10-engineering-practices/04_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: From Notebook to Project

In Module 9 you built a logistic regression classifier for penguin sex, entirely in a notebook. It worked, but try to use it for anything else and the problems appear:

- To predict for new penguins, you'd have to re-run the whole notebook, top to bottom.
- The settings (features, seed, split) are scattered through the cells.
- Nothing records which data, settings and library versions produced the 91% accuracy.
- There are checks, but they're cells you have to remember to run.

In this project you'll turn it into a small, proper project, the way you'd structure it for a team or for yourself six months from now:

```
penguin-sex/
├── pyproject.toml
├── README.md
├── configs/                  experiment settings as JSON
├── penguin_sex/              the package
│   ├── config.py             settings, validated
│   ├── data.py               loading, checking, splitting, features
│   ├── model.py              logistic regression with Newton's method
│   ├── persistence.py        saving and loading a trained model
│   └── cli.py                `train` and `predict` commands
└── tests/                    a pytest suite that runs in a couple of seconds
```

Then you'll use it like a real tool: train from the command line, inspect the saved run, predict for new penguins, and run a small experiment across random seeds.

## Setup

The project is written into `work/penguin-sex`. Each run of this notebook starts from an empty folder.

In [1]:
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path("work/penguin-sex")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
for folder in ["penguin_sex", "tests", "configs"]:
    (PROJECT / folder).mkdir(parents=True)

DATA_URL = "https://raw.githubusercontent.com/anupbagale/python-for-ai-ml/main/05-files-and-data/data/penguins.csv"
local = Path("data/penguins.csv")
DATA = str(local.resolve()) if local.exists() else DATA_URL     # absolute, so it works from any folder
print("project:", PROJECT)

project: work/penguin-sex


## Step 1: Configuration

Every setting that affects the result goes in one frozen dataclass, validated as soon as it's created. JSON has no tuples, so a feature list read from a file is converted to a tuple, which keeps the config immutable.

In [2]:
%%writefile work/penguin-sex/penguin_sex/config.py
"""Training settings, loaded from JSON and checked before anything runs."""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, fields
from pathlib import Path

MEASUREMENTS = ("bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g")


@dataclass(frozen=True)
class TrainConfig:
    features: tuple[str, ...] = MEASUREMENTS
    use_species: bool = True
    test_fraction: float = 0.25
    seed: int = 0
    l2: float = 0.0
    max_iter: int = 25

    def __post_init__(self):
        object.__setattr__(self, "features", tuple(self.features))   # JSON gives a list
        unknown = set(self.features) - set(MEASUREMENTS)
        if not self.features or unknown:
            raise ValueError(f"features must be a non-empty subset of {MEASUREMENTS}, got {self.features}")
        if not 0 < self.test_fraction < 1:
            raise ValueError(f"test_fraction must be between 0 and 1, got {self.test_fraction}")
        if self.l2 < 0:
            raise ValueError(f"l2 must be at least 0, got {self.l2}")
        if self.max_iter < 1:
            raise ValueError(f"max_iter must be at least 1, got {self.max_iter}")

    @classmethod
    def from_json(cls, path) -> "TrainConfig":
        values = json.loads(Path(path).read_text())
        unknown = set(values) - {f.name for f in fields(cls)}
        if unknown:
            raise KeyError(f"unknown config keys in {path}: {sorted(unknown)}")
        return cls(**values)

    def to_dict(self) -> dict:
        values = asdict(self)
        values["features"] = list(self.features)
        return values

Writing work/penguin-sex/penguin_sex/config.py


## Step 2: Data

`data.py` holds everything between the CSV file and the feature matrix:

- **`load_penguins`** checks the columns and values, drops incomplete rows, and **logs** how many it dropped, so silent data loss shows up in the training log.
- **`fingerprint`** hashes the loaded table, as in lesson section 8.
- **`stratified_split`** keeps the same mix of species and sex in both sets, using a `Generator` made from the seed.
- **`FeatureBuilder`** learns the standardisation from the training data and can be saved and restored, because a model is useless without the exact preprocessing it was trained with.

In [3]:
%%writefile work/penguin-sex/penguin_sex/data.py
"""Loading, checking and splitting the penguin data, and building features."""
from __future__ import annotations

import hashlib
import logging

import numpy as np
import pandas as pd

from .config import MEASUREMENTS

log = logging.getLogger(__name__)

SPECIES = ("Adelie", "Chinstrap", "Gentoo")


class DataError(ValueError):
    """The input data doesn't have the columns or values the model needs."""


def load_penguins(source, require_sex: bool = True) -> pd.DataFrame:
    df = pd.read_csv(source)
    required = ["species", *MEASUREMENTS] + (["sex"] if require_sex else [])
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise DataError(f"missing columns: {missing}")
    unknown = set(df["species"].dropna()) - set(SPECIES)
    if unknown:
        raise DataError(f"unknown species: {sorted(unknown)}")

    before = len(df)
    df = df.dropna(subset=required).reset_index(drop=True)
    if len(df) < before:
        log.warning("dropped %d of %d rows with missing values", before - len(df), before)
    if require_sex:
        bad = set(df["sex"]) - {"female", "male"}
        if bad:
            raise DataError(f"sex must be 'female' or 'male', found {sorted(bad)}")
    log.info("loaded %d penguins", len(df))
    return df


def fingerprint(df: pd.DataFrame) -> str:
    """A short hash of the table's contents: changes if any value changes."""
    return hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()[:16]


def stratified_split(df: pd.DataFrame, test_fraction: float, seed: int):
    """Split so every species-and-sex group is divided in the same proportion."""
    rng = np.random.default_rng(seed)
    test_rows = []
    for _, group in df.groupby(["species", "sex"], sort=True):
        rows = rng.permutation(group.index.to_numpy())
        test_rows.extend(rows[:round(len(rows) * test_fraction)])
    is_test = df.index.isin(test_rows)
    return df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)


def labels(df: pd.DataFrame) -> np.ndarray:
    return (df["sex"] == "male").to_numpy(dtype=float)


class FeatureBuilder:
    """Standardised measurements, plus 0/1 columns for Chinstrap and Gentoo if use_species is set."""

    def __init__(self, features=MEASUREMENTS, use_species: bool = True):
        self.features = list(features)
        self.use_species = use_species
        self.mean_ = None
        self.std_ = None

    def fit(self, df: pd.DataFrame) -> "FeatureBuilder":
        values = df[self.features].to_numpy(dtype=float)
        self.mean_ = values.mean(axis=0)
        std = values.std(axis=0)
        self.std_ = np.where(std == 0, 1.0, std)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        if self.mean_ is None:
            raise RuntimeError("call fit() before transform()")
        X = (df[self.features].to_numpy(dtype=float) - self.mean_) / self.std_
        if self.use_species:
            indicators = [(df["species"] == sp).to_numpy(dtype=float) for sp in SPECIES[1:]]
            X = np.column_stack([X, *indicators])
        return X

    @property
    def names(self) -> list:
        return self.features + ([f"is_{sp}" for sp in SPECIES[1:]] if self.use_species else [])

    def to_dict(self) -> dict:
        return {"features": self.features, "use_species": self.use_species,
                "mean": self.mean_.tolist(), "std": self.std_.tolist()}

    @classmethod
    def from_dict(cls, values: dict) -> "FeatureBuilder":
        builder = cls(values["features"], values["use_species"])
        builder.mean_ = np.array(values["mean"])
        builder.std_ = np.array(values["std"])
        return builder

Writing work/penguin-sex/penguin_sex/data.py


## Step 3: The model

The model from Module 9, as a class with the familiar `fit` / `predict_proba` / `predict` interface. Training uses Newton's method, which needs no learning rate and converges in a handful of steps. There's one addition: an optional **L2 penalty**, `l2 · ‖w‖² / 2`, which pulls the weights towards zero. It adds `l2 · w` to the gradient and `l2` to the diagonal of the Hessian (but not for the bias).

In [4]:
%%writefile work/penguin-sex/penguin_sex/model.py
"""Binary logistic regression trained with Newton's method."""
from __future__ import annotations

import logging

import numpy as np

log = logging.getLogger(__name__)


def sigmoid(z: np.ndarray) -> np.ndarray:
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1 / (1 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1 + ez)
    return out


class LogisticRegression:
    def __init__(self, l2: float = 0.0, max_iter: int = 25, tol: float = 1e-10):
        self.l2 = l2
        self.max_iter = max_iter
        self.tol = tol
        self.coef_ = None
        self.intercept_ = None
        self.n_iter_ = 0

    def _penalty(self, d: int) -> np.ndarray:
        return np.r_[np.full(d, self.l2), 0.0]          # no penalty on the bias

    def loss_and_grad(self, theta: np.ndarray, X: np.ndarray, y: np.ndarray):
        """Mean log loss plus the L2 penalty, and its gradient. theta = weights followed by the bias."""
        A = np.column_stack([X, np.ones(len(X))])
        z = A @ theta
        penalty = self._penalty(X.shape[1])
        loss = np.mean(np.logaddexp(0, z) - y * z) + 0.5 * np.sum(penalty * theta ** 2)
        grad = A.T @ (sigmoid(z) - y) / len(y) + penalty * theta
        return float(loss), grad

    def fit(self, X, y) -> "LogisticRegression":
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        if X.ndim != 2 or len(X) != len(y):
            raise ValueError(f"X must be 2D with one row per label; got X {X.shape} and y {y.shape}")
        if not np.isin(y, [0, 1]).all():
            raise ValueError("labels must be 0 or 1")

        A = np.column_stack([X, np.ones(len(X))])
        penalty = self._penalty(X.shape[1])
        theta = np.zeros(A.shape[1])
        for iteration in range(1, self.max_iter + 1):
            p = sigmoid(A @ theta)
            grad = A.T @ (p - y) / len(y) + penalty * theta
            hessian = A.T @ (A * (p * (1 - p))[:, None]) / len(y) + np.diag(penalty)
            step = np.linalg.solve(hessian, grad)
            theta -= step
            log.debug("iteration %d: largest step %.2e", iteration, np.abs(step).max())
            if np.abs(step).max() < self.tol:
                break
        else:
            log.warning("did not converge in %d iterations", self.max_iter)
        self.n_iter_ = iteration
        self.coef_, self.intercept_ = theta[:-1], float(theta[-1])
        return self

    def predict_proba(self, X) -> np.ndarray:
        if self.coef_ is None:
            raise RuntimeError("call fit() before predicting")
        return sigmoid(np.asarray(X, dtype=float) @ self.coef_ + self.intercept_)

    def predict(self, X, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(X) >= threshold).astype(int)

Writing work/penguin-sex/penguin_sex/model.py


The `for ... else` in `fit` is a less common Python feature: the `else` block runs only if the loop finished **without** `break`. Here that means Newton's method used every iteration without converging, which deserves a warning in the log.

## Step 4: Saving and loading

A first version that saves the model's weights to JSON. JSON is readable, diffable and safe to load, unlike `pickle`, which can run arbitrary code when loading a file from an untrusted source.

In [5]:
%%writefile work/penguin-sex/penguin_sex/persistence.py
"""Save and load a trained model as JSON."""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

from .data import FeatureBuilder
from .model import LogisticRegression


def save_model(path, builder: FeatureBuilder, model: LogisticRegression) -> None:
    record = {"coef": model.coef_.tolist(), "intercept": model.intercept_, "l2": model.l2}
    Path(path).write_text(json.dumps(record, indent=2))


def load_model(path):
    record = json.loads(Path(path).read_text())
    model = LogisticRegression(l2=record["l2"])
    model.coef_ = np.array(record["coef"])
    model.intercept_ = record["intercept"]
    return FeatureBuilder(), model

Writing work/penguin-sex/penguin_sex/persistence.py


In [6]:
%%writefile work/penguin-sex/penguin_sex/__init__.py
"""Predict a penguin's sex from its measurements."""
__version__ = "0.1.0"

Writing work/penguin-sex/penguin_sex/__init__.py


## Step 5: Tests

Now the safety net. The tests use a small **synthetic** dataset built in `conftest.py`, a special file pytest loads automatically, whose fixtures every test file can use. Synthetic data keeps the tests fast, independent of the internet, and in full control: we know exactly what the right answers are.

In [7]:
%%writefile work/penguin-sex/tests/conftest.py
import logging

import numpy as np
import pandas as pd
import pytest

MEANS = {"Adelie": (38.8, 18.3, 190.0, 3700.0), "Chinstrap": (48.8, 18.4, 196.0, 3730.0),
         "Gentoo": (47.5, 15.0, 217.0, 5080.0)}
SPREAD = (1.0, 0.4, 3.0, 150.0)


@pytest.fixture
def penguins_df():
    """60 penguins per species; males are larger on every measurement."""
    rng = np.random.default_rng(0)
    rows = []
    for species, means in MEANS.items():
        for sex, shift in [("female", -1.0), ("male", 1.0)]:
            for _ in range(30):
                values = [m + shift * s + rng.normal(0, s) for m, s in zip(means, SPREAD)]
                rows.append({"species": species, "island": "Biscoe", "sex": sex,
                             **dict(zip(["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"], values))})
    return pd.DataFrame(rows)


@pytest.fixture
def penguins_csv(tmp_path, penguins_df):
    path = tmp_path / "penguins.csv"
    penguins_df.to_csv(path, index=False)
    return path


@pytest.fixture(autouse=True)
def reset_package_logger():
    """The CLI configures the package's logger. Undo that after every test, so tests can't affect each other."""
    yield
    logger = logging.getLogger("penguin_sex")
    for handler in list(logger.handlers):
        logger.removeHandler(handler)
        handler.close()
    logger.setLevel(logging.NOTSET)
    logger.propagate = True

Writing work/penguin-sex/tests/conftest.py


The last fixture deserves a word. `autouse=True` makes it run around **every** test without being asked for. The code before `yield` would run before each test; the code after runs after it, even if the test failed. It's there because loggers are global: the CLI tests (coming in Step 6) configure the package's logger and switch off propagation, and without this reset, a later test that checks log messages could pass or fail depending on which tests happened to run before it. That's exactly what happened while this project was being written: the logging test passed on one pytest version and failed on another. Tests that depend on each other's leftovers are some of the hardest bugs to track down, so any test that changes global state should put it back.

In [8]:
%%writefile work/penguin-sex/tests/test_config.py
import json

import pytest

from penguin_sex.config import MEASUREMENTS, TrainConfig


def test_defaults_are_valid():
    config = TrainConfig()
    assert config.features == MEASUREMENTS and config.use_species


def test_from_json_converts_features_to_a_tuple(tmp_path):
    path = tmp_path / "c.json"
    path.write_text(json.dumps({"features": ["body_mass_g"], "seed": 3}))
    config = TrainConfig.from_json(path)
    assert config.features == ("body_mass_g",)
    assert config.seed == 3


def test_unknown_key_is_rejected(tmp_path):
    path = tmp_path / "c.json"
    path.write_text(json.dumps({"learning_rate": 0.1}))
    with pytest.raises(KeyError, match="learning_rate"):
        TrainConfig.from_json(path)


@pytest.mark.parametrize("bad", [{"test_fraction": 1.0}, {"l2": -1.0}, {"max_iter": 0},
                                 {"features": []}, {"features": ["beak_colour"]}])
def test_invalid_values_are_rejected(bad):
    with pytest.raises(ValueError):
        TrainConfig(**bad)

Writing work/penguin-sex/tests/test_config.py


In [9]:
%%writefile work/penguin-sex/tests/test_data.py
import logging

import numpy as np
import pytest

from penguin_sex.data import DataError, FeatureBuilder, labels, load_penguins, stratified_split


def test_missing_column_is_reported(tmp_path, penguins_df):
    path = tmp_path / "bad.csv"
    penguins_df.drop(columns="body_mass_g").to_csv(path, index=False)
    with pytest.raises(DataError, match="body_mass_g"):
        load_penguins(path)


def test_incomplete_rows_are_dropped_and_logged(tmp_path, penguins_df, caplog):
    penguins_df.loc[[0, 5], "bill_depth_mm"] = np.nan
    path = tmp_path / "gaps.csv"
    penguins_df.to_csv(path, index=False)
    with caplog.at_level(logging.WARNING):
        df = load_penguins(path)
    assert len(df) == len(penguins_df) - 2
    assert "dropped 2" in caplog.text


def test_unknown_species_is_rejected(tmp_path, penguins_df):
    penguins_df.loc[0, "species"] = "Emperor"
    path = tmp_path / "emperor.csv"
    penguins_df.to_csv(path, index=False)
    with pytest.raises(DataError, match="Emperor"):
        load_penguins(path)


def test_split_is_stratified_disjoint_and_complete(penguins_df):
    train, test = stratified_split(penguins_df, test_fraction=0.25, seed=0)
    assert len(train) + len(test) == len(penguins_df)
    assert test.groupby(["species", "sex"]).size().tolist() == [8] * 6     # round(30 * 0.25)
    keys = lambda df: set(map(tuple, df[["bill_length_mm", "body_mass_g"]].to_numpy()))
    assert not keys(train) & keys(test)


def test_split_is_reproducible(penguins_df):
    a = stratified_split(penguins_df, 0.25, seed=4)[1]
    b = stratified_split(penguins_df, 0.25, seed=4)[1]
    c = stratified_split(penguins_df, 0.25, seed=5)[1]
    assert a.equals(b)
    assert not a.equals(c)


def test_features_use_training_statistics(penguins_df):
    builder = FeatureBuilder().fit(penguins_df)
    X = builder.transform(penguins_df)
    assert X.shape == (len(penguins_df), 6)
    assert X[:, :4].mean(axis=0) == pytest.approx(0, abs=1e-12)
    assert set(np.unique(X[:, 4:])) == {0.0, 1.0}
    assert builder.names[-2:] == ["is_Chinstrap", "is_Gentoo"]


def test_feature_builder_round_trips(penguins_df):
    builder = FeatureBuilder(["body_mass_g"], use_species=False).fit(penguins_df)
    restored = FeatureBuilder.from_dict(builder.to_dict())
    assert np.array_equal(restored.transform(penguins_df), builder.transform(penguins_df))


def test_labels_are_one_for_males(penguins_df):
    y = labels(penguins_df)
    assert y.sum() == (penguins_df["sex"] == "male").sum()

Writing work/penguin-sex/tests/test_data.py


In [10]:
%%writefile work/penguin-sex/tests/test_model.py
import numpy as np
import pytest

from penguin_sex.model import LogisticRegression


@pytest.fixture
def blobs():
    rng = np.random.default_rng(1)
    X = np.vstack([rng.normal(-1, 1, size=(100, 3)), rng.normal(1, 1, size=(100, 3))])
    y = np.r_[np.zeros(100), np.ones(100)]
    return X, y


@pytest.mark.parametrize("l2", [0.0, 0.5])
def test_gradient_matches_numerical(blobs, l2):
    X, y = blobs
    model = LogisticRegression(l2=l2)
    theta = np.random.default_rng(2).normal(size=4)
    _, grad = model.loss_and_grad(theta, X, y)
    h = 1e-6
    numerical = np.array([
        (model.loss_and_grad(theta + h * e, X, y)[0] - model.loss_and_grad(theta - h * e, X, y)[0]) / (2 * h)
        for e in np.eye(4)
    ])
    assert grad == pytest.approx(numerical, abs=1e-7)


def test_learns_an_easy_problem(blobs):
    X, y = blobs
    model = LogisticRegression().fit(X, y)
    assert np.mean(model.predict(X) == y) > 0.9
    assert model.n_iter_ < 25


def test_probabilities_are_valid(blobs):
    X, y = blobs
    p = LogisticRegression().fit(X, y).predict_proba(X * 1000)        # extreme inputs
    assert np.all((p >= 0) & (p <= 1)) and not np.any(np.isnan(p))


def test_l2_shrinks_the_weights(blobs):
    X, y = blobs
    plain = LogisticRegression(l2=0.0).fit(X, y)
    shrunk = LogisticRegression(l2=1.0).fit(X, y)
    assert np.linalg.norm(shrunk.coef_) < np.linalg.norm(plain.coef_)


@pytest.mark.parametrize("X, y", [(np.ones((3, 2)), np.array([0, 1])), (np.ones((2, 2)), np.array([0, 2]))])
def test_bad_training_data_is_rejected(X, y):
    with pytest.raises(ValueError):
        LogisticRegression().fit(X, y)


def test_predict_before_fit_raises():
    with pytest.raises(RuntimeError):
        LogisticRegression().predict(np.ones((1, 2)))

Writing work/penguin-sex/tests/test_model.py


In [11]:
%%writefile work/penguin-sex/tests/test_persistence.py
import numpy as np

from penguin_sex.data import FeatureBuilder, labels
from penguin_sex.model import LogisticRegression
from penguin_sex.persistence import load_model, save_model


def test_saved_model_predicts_the_same(tmp_path, penguins_df):
    builder = FeatureBuilder().fit(penguins_df)
    model = LogisticRegression().fit(builder.transform(penguins_df), labels(penguins_df))
    save_model(tmp_path / "model.json", builder, model)

    loaded_builder, loaded_model = load_model(tmp_path / "model.json")
    before = model.predict_proba(builder.transform(penguins_df))
    after = loaded_model.predict_proba(loaded_builder.transform(penguins_df))
    assert np.array_equal(before, after)

Writing work/penguin-sex/tests/test_persistence.py


A note on the gradient test: it's Module 9's gradient check, now **permanent**. If anyone changes the loss or the penalty later and gets the gradient wrong, the suite fails straight away. It's parametrised with and without the L2 penalty, because that's exactly the kind of addition that tends to get the gradient right and the Hessian, or the bias, wrong.

Run the suite:

In [12]:
!cd {PROJECT} && {sys.executable} -m pytest -q --tb=short -p no:cacheprovider

.

..................

.....F                                                [100%]
=================================== FAILURES ===================================
______________________ test_saved_model_predicts_the_same ______________________
tests/test_persistence.py:15: in test_saved_model_predicts_the_same
    after = loaded_model.predict_proba(loaded_builder.transform(penguins_df))
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
penguin_sex/data.py:81: in transform
    raise RuntimeError("call fit() before transform()")
E   RuntimeError: call fit() before transform()
=========================== short test summary info ============================
FAILED tests/test_persistence.py::test_saved_model_predicts_the_same - Runtim...
1 failed, 24 passed in 0.12s


One failure, and it's a real one: the saved model can't make predictions after loading, because `save_model` stored the weights but **not the preprocessing**. The loaded `FeatureBuilder` was never fitted, so it doesn't know the training means and standard deviations.

This is one of the most common bugs in deployed ML. The weights only make sense together with the exact transformation that produced their inputs. If they're saved separately, sooner or later someone loads a model with the wrong scaler, and it doesn't crash: it just predicts badly. (Two optional flags in that command: `--tb=short` keeps tracebacks brief, and `-p no:cacheprovider` stops pytest from writing a `.pytest_cache` folder.)

Fix it by saving both, together, in one file. A `format` number also makes it possible to change the file layout later without misreading old files:

In [13]:
%%writefile work/penguin-sex/penguin_sex/persistence.py
"""Save and load a trained model, together with its preprocessing, as JSON."""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

from .data import FeatureBuilder
from .model import LogisticRegression

FORMAT = 1


def save_model(path, builder: FeatureBuilder, model: LogisticRegression) -> None:
    record = {
        "format": FORMAT,
        "features": builder.to_dict(),
        "model": {"coef": model.coef_.tolist(), "intercept": model.intercept_, "l2": model.l2},
    }
    Path(path).write_text(json.dumps(record, indent=2))


def load_model(path):
    record = json.loads(Path(path).read_text())
    if record.get("format") != FORMAT:
        raise ValueError(f"{path} has format {record.get('format')}, expected {FORMAT}")
    model = LogisticRegression(l2=record["model"]["l2"])
    model.coef_ = np.array(record["model"]["coef"])
    model.intercept_ = record["model"]["intercept"]
    return FeatureBuilder.from_dict(record["features"]), model

Overwriting work/penguin-sex/penguin_sex/persistence.py


In [14]:
!cd {PROJECT} && {sys.executable} -m pytest -q --tb=short -p no:cacheprovider

.............

............                                                [100%]
25 passed in 0.08s


All green, and the test stays in the suite, so this bug can't come back.

## Step 6: The command-line interface

The last module ties the pieces together into two commands:

- **`train`** reads a config and a CSV, trains, and saves a **run folder** with the model, a metrics file and a log. It refuses to overwrite an existing run unless told to.
- **`predict`** loads a saved model and writes predictions for a CSV of new penguins.

`argparse` subcommands give each its own options and help text. Logging goes to the screen and to `train.log` in the run folder; `--quiet` shows only errors on screen, while the log file still gets everything.

In [15]:
%%writefile work/penguin-sex/penguin_sex/cli.py
"""Command-line interface: python -m penguin_sex.cli {train,predict} ..."""
from __future__ import annotations

import argparse
import json
import logging
import platform
import sys
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from . import __version__
from .config import TrainConfig
from .data import FeatureBuilder, fingerprint, labels, load_penguins, stratified_split
from .model import LogisticRegression
from .persistence import load_model, save_model

log = logging.getLogger("penguin_sex.cli")


def setup_logging(log_file=None, quiet: bool = False) -> None:
    """Configure the package's logger: once per command, no duplicate handlers."""
    logger = logging.getLogger("penguin_sex")
    for handler in list(logger.handlers):
        logger.removeHandler(handler)
        handler.close()
    logger.setLevel(logging.DEBUG)
    logger.propagate = False
    screen = logging.StreamHandler(sys.stdout)
    screen.setLevel(logging.ERROR if quiet else logging.INFO)
    screen.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
    logger.addHandler(screen)
    if log_file is not None:
        file = logging.FileHandler(log_file, mode="w")
        file.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s"))
        logger.addHandler(file)


def accuracy(model, X, y) -> float:
    return float(np.mean(model.predict(X) == y))


def train(args) -> dict:
    config = TrainConfig.from_json(args.config) if args.config else TrainConfig()
    if args.seed is not None:
        config = replace(config, seed=args.seed)

    out = Path(args.out)
    if (out / "metrics.json").exists() and not args.overwrite:
        raise FileExistsError(f"{out} already contains a run; use --overwrite to replace it")
    out.mkdir(parents=True, exist_ok=True)
    setup_logging(out / "train.log", args.quiet)
    log.info("config: %s", config.to_dict())

    df = load_penguins(args.data)
    train_df, test_df = stratified_split(df, config.test_fraction, config.seed)
    builder = FeatureBuilder(config.features, config.use_species).fit(train_df)
    X_train, y_train = builder.transform(train_df), labels(train_df)
    X_test, y_test = builder.transform(test_df), labels(test_df)

    model = LogisticRegression(l2=config.l2, max_iter=config.max_iter).fit(X_train, y_train)
    metrics = {
        "train_accuracy": accuracy(model, X_train, y_train),
        "test_accuracy": accuracy(model, X_test, y_test),
        "n_train": len(train_df),
        "n_test": len(test_df),
        "newton_iterations": model.n_iter_,
        "weights": dict(zip(builder.names + ["bias"], map(float, np.r_[model.coef_, model.intercept_]))),
    }
    log.info("train accuracy %.3f, test accuracy %.3f (%d test penguins)",
             metrics["train_accuracy"], metrics["test_accuracy"], metrics["n_test"])

    save_model(out / "model.json", builder, model)
    record = {
        "config": config.to_dict(),
        "metrics": metrics,
        "data": {"source": str(args.data), "rows": len(df), "fingerprint": fingerprint(df)},
        "environment": {"penguin_sex": __version__, "python": platform.python_version(),
                        "numpy": np.__version__, "pandas": pd.__version__},
        "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    (out / "metrics.json").write_text(json.dumps(record, indent=2))
    log.info("saved run to %s", out)
    return record


def predict(args) -> pd.DataFrame:
    setup_logging(quiet=args.quiet)
    builder, model = load_model(args.model)
    df = load_penguins(args.data, require_sex=False)
    result = df.copy()
    result["p_male"] = model.predict_proba(builder.transform(df)).round(4)
    result["predicted_sex"] = np.where(result["p_male"] >= 0.5, "male", "female")
    result.to_csv(args.out, index=False)
    log.info("wrote %d predictions to %s", len(result), args.out)
    return result


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(prog="penguin_sex", description="Predict a penguin's sex from its measurements.")
    commands = parser.add_subparsers(dest="command", required=True)

    t = commands.add_parser("train", help="train a model and save a run folder")
    t.add_argument("--data", required=True, help="CSV file or URL with labelled penguins")
    t.add_argument("--config", type=Path, help="JSON config file (defaults are used if omitted)")
    t.add_argument("--seed", type=int, help="override the config's random seed")
    t.add_argument("--out", type=Path, required=True, help="run folder to create")
    t.add_argument("--overwrite", action="store_true", help="replace an existing run")
    t.add_argument("--quiet", action="store_true", help="only show errors on screen (the log file still gets everything)")
    t.set_defaults(func=train)

    p = commands.add_parser("predict", help="predict for new penguins with a saved model")
    p.add_argument("--model", type=Path, required=True, help="model.json from a run folder")
    p.add_argument("--data", required=True, help="CSV file with measurements and species")
    p.add_argument("--out", type=Path, required=True, help="CSV file to write")
    p.add_argument("--quiet", action="store_true", help="only show errors on screen")
    p.set_defaults(func=predict)
    return parser


def main(argv=None):
    args = build_parser().parse_args(argv)
    return args.func(args)


if __name__ == "__main__":
    main()

Writing work/penguin-sex/penguin_sex/cli.py


And an end-to-end test of both commands, run on the synthetic data in a temporary folder. Because `main` accepts a list of arguments, the test calls it directly, exactly as the command line would.

In [16]:
%%writefile work/penguin-sex/tests/test_cli.py
import json

import pandas as pd
import pytest

from penguin_sex.cli import main


def test_train_then_predict(tmp_path, penguins_csv):
    run = tmp_path / "run"
    record = main(["train", "--data", str(penguins_csv), "--out", str(run), "--quiet"])

    assert {p.name for p in run.iterdir()} == {"model.json", "metrics.json", "train.log"}
    saved = json.loads((run / "metrics.json").read_text())
    assert saved["metrics"] == record["metrics"]
    assert saved["metrics"]["test_accuracy"] > 0.8
    assert "test accuracy" in (run / "train.log").read_text()

    new = pd.read_csv(penguins_csv).drop(columns="sex").head(10)
    new.to_csv(tmp_path / "new.csv", index=False)
    main(["predict", "--model", str(run / "model.json"), "--data", str(tmp_path / "new.csv"),
          "--out", str(tmp_path / "pred.csv"), "--quiet"])
    predictions = pd.read_csv(tmp_path / "pred.csv")
    assert len(predictions) == 10
    assert predictions["p_male"].between(0, 1).all()
    assert set(predictions["predicted_sex"]) <= {"male", "female"}


def test_refuses_to_overwrite_a_run(tmp_path, penguins_csv):
    args = ["train", "--data", str(penguins_csv), "--out", str(tmp_path / "run"), "--quiet"]
    main(args)
    with pytest.raises(FileExistsError):
        main(args)
    main(args + ["--overwrite"])


def test_same_seed_same_result(tmp_path, penguins_csv):
    runs = [main(["train", "--data", str(penguins_csv), "--out", str(tmp_path / name), "--seed", "3", "--quiet"])
            for name in ["a", "b"]]
    assert runs[0]["metrics"] == runs[1]["metrics"]

Writing work/penguin-sex/tests/test_cli.py


In [17]:
%%writefile work/penguin-sex/pyproject.toml
[project]
name = "penguin-sex"
version = "0.1.0"
description = "Predict a penguin's sex from its measurements"
requires-python = ">=3.10"
dependencies = ["numpy>=1.26", "pandas>=2.0"]

[project.optional-dependencies]
dev = ["pytest>=8"]

[project.scripts]
penguin-sex = "penguin_sex.cli:main"

[tool.pytest.ini_options]
pythonpath = ["."]
testpaths = ["tests"]

Writing work/penguin-sex/pyproject.toml


`[project.scripts]` means that after `pip install -e .`, the project gets a real terminal command, `penguin-sex train ...`, instead of `python -m penguin_sex.cli train ...`. `[project.optional-dependencies]` lists tools only developers need, installed with `pip install -e ".[dev]"`.

The full suite:

In [18]:
!cd {PROJECT} && {sys.executable} -m pytest -q --tb=short -p no:cacheprovider

..

...............

...........                                             [100%]
28 passed in 0.15s


## Step 7: Use it

Two configs: the full model, and one without species, the comparison Module 9 couldn't settle.

In [19]:
(PROJECT / "configs/default.json").write_text(json.dumps({"use_species": True}, indent=2))
(PROJECT / "configs/no_species.json").write_text(json.dumps({"use_species": False}, indent=2))
!cd {PROJECT} && {sys.executable} -m penguin_sex.cli train --help

usage: penguin_sex train [-h] --data DATA [--config CONFIG] [--seed SEED]
                         --out OUT [--overwrite] [--quiet]

options:
  -h, --help       show this help message and exit
  --data DATA      CSV file or URL with labelled penguins
  --config CONFIG  JSON config file (defaults are used if omitted)
  --seed SEED      override the config's random seed
  --out OUT        run folder to create
  --overwrite      replace an existing run
  --quiet          only show errors on screen (the log file still gets
                   everything)


In [20]:
!cd {PROJECT} && {sys.executable} -m penguin_sex.cli train --data {DATA} --config configs/default.json --out runs/default

INFO config: {'features': ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'], 'use_species': True, 'test_fraction': 0.25, 'seed': 0, 'l2': 0.0, 'max_iter': 25}
WARNING dropped 11 of 344 rows with missing values
INFO loaded 333 penguins
INFO train accuracy 0.929, test accuracy 0.901 (81 test penguins)
INFO saved run to runs/default


The warning about dropped rows is the data check doing its job: 11 penguins have no recorded sex (or missing measurements) and can't be used for training. The log says so instead of the count quietly shrinking.

Everything about the run is in its folder:

In [21]:
run = PROJECT / "runs/default"
print(sorted(p.name for p in run.iterdir()))
record = json.loads((run / "metrics.json").read_text())
print(json.dumps({k: record[k] for k in ["config", "data", "environment"]}, indent=2))
print("test accuracy:", round(record["metrics"]["test_accuracy"], 3))

['metrics.json', 'model.json', 'train.log']
{
  "config": {
    "features": [
      "bill_length_mm",
      "bill_depth_mm",
      "flipper_length_mm",
      "body_mass_g"
    ],
    "use_species": true,
    "test_fraction": 0.25,
    "seed": 0,
    "l2": 0.0,
    "max_iter": 25
  },
  "data": {
    "source": "/content/data/penguins.csv",
    "rows": 333,
    "fingerprint": "42f2cbca76cc5f79"
  },
  "environment": {
    "penguin_sex": "0.1.0",
    "python": "3.13.18",
    "numpy": "2.5.3",
    "pandas": "3.0.5"
  }
}
test accuracy: 0.901


In [22]:
print((run / "train.log").read_text())

2026-11-05 05:24:46,247 INFO penguin_sex.cli: config: {'features': ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'], 'use_species': True, 'test_fraction': 0.25, 'seed': 0, 'l2': 0.0, 'max_iter': 25}
2026-11-05 05:24:46,251 WARNING penguin_sex.data: dropped 11 of 344 rows with missing values
2026-11-05 05:24:46,251 INFO penguin_sex.data: loaded 333 penguins
2026-11-05 05:24:46,256 DEBUG penguin_sex.model: iteration 1: largest step 1.88e+00
2026-11-05 05:24:46,256 DEBUG penguin_sex.model: iteration 2: largest step 1.62e+00
2026-11-05 05:24:46,256 DEBUG penguin_sex.model: iteration 3: largest step 1.83e+00
2026-11-05 05:24:46,256 DEBUG penguin_sex.model: iteration 4: largest step 1.68e+00
2026-11-05 05:24:46,256 DEBUG penguin_sex.model: iteration 5: largest step 1.02e+00
2026-11-05 05:24:46,256 DEBUG penguin_sex.model: iteration 6: largest step 2.28e-01
2026-11-05 05:24:46,257 DEBUG penguin_sex.model: iteration 7: largest step 7.70e-03
2026-11-05 05:24:46,257 DEBUG 

The log file has more detail than the screen: timestamps, which module wrote each line, and the `DEBUG` messages from every Newton iteration, showing the step size collapsing as it converges. If a run ever behaves strangely, this is where you'd look first.

Trying to train into the same folder again is refused, so a finished result can't be lost by re-running a cell:

In [23]:
!cd {PROJECT} && {sys.executable} -m penguin_sex.cli train --data {DATA} --out runs/default --quiet 2>&1 | tail -1

FileExistsError: runs/default already contains a run; use --overwrite to replace it


### Predicting for new penguins

A field team measures five penguins but can't tell their sex. Their CSV has the measurements and species, and no `sex` column:

In [24]:
new_penguins = pd.DataFrame({
    "species": ["Adelie", "Adelie", "Gentoo", "Chinstrap", "Gentoo"],
    "bill_length_mm": [36.5, 41.1, 45.2, 51.3, 49.9],
    "bill_depth_mm": [17.0, 19.6, 13.9, 19.9, 16.1],
    "flipper_length_mm": [185, 196, 211, 198, 223],
    "body_mass_g": [3300, 4300, 4650, 3900, 5600],
})
new_penguins.to_csv(PROJECT / "new_penguins.csv", index=False)
!cd {PROJECT} && {sys.executable} -m penguin_sex.cli predict --model runs/default/model.json --data new_penguins.csv --out predictions.csv
pd.read_csv(PROJECT / "predictions.csv")

INFO loaded 5 penguins
INFO wrote 5 predictions to predictions.csv


,species,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,p_male,predicted_sex
0,Adelie,36.5,17.0,185,3300,0.0014,female
1,Adelie,41.1,19.6,196,4300,0.9995,male
2,Gentoo,45.2,13.9,211,4650,0.0029,female
3,Chinstrap,51.3,19.9,198,3900,0.9922,male
4,Gentoo,49.9,16.1,223,5600,0.9993,male


The model only needed `model.json`: weights, feature list and standardisation together, the fix from Step 5. Notice the Gentoo at 4,650 g is predicted female even though it's heavier than every Adelie here. The species columns let the model judge size **relative to its species**, as Module 9 found.

## Step 8: A small experiment

Module 9 couldn't tell whether adding species helps, because one test set of 81 penguins is too small. With a CLI, repeating the comparison over many different random splits is a loop. Calling `main` directly (instead of starting a new Python process each time) is faster:

In [25]:
sys.path.insert(0, str(PROJECT))
from penguin_sex.cli import main

rows = []
for seed in range(20):
    for name in ["default", "no_species"]:
        record = main(["train", "--data", DATA, "--config", str(PROJECT / f"configs/{name}.json"),
                       "--seed", str(seed), "--out", str(PROJECT / f"runs/{name}_seed{seed}"), "--quiet"])
        rows.append({"config": name, "seed": seed, "test_accuracy": record["metrics"]["test_accuracy"]})

results = pd.DataFrame(rows).pivot(index="seed", columns="config", values="test_accuracy")
results["difference"] = results["default"] - results["no_species"]
results.describe().loc[["mean", "std", "min", "max"]].round(3)

config,default,no_species,difference
mean,0.915,0.905,0.010
std,0.024,0.023,0.026
min,0.852,0.852,-0.049
max,0.951,0.938,0.049


In [26]:
better = (results["difference"] > 0).sum()
same = (results["difference"] == 0).sum()
print(f"species helped on {better} of 20 splits, tied on {same}, hurt on {20 - better - same}")
n_test = record["metrics"]["n_test"]
gain = results["difference"].mean()
print(f"average gain: {gain:+.3f}, about {gain * n_test:.1f} penguins per test set of {n_test}")

species helped on 11 of 20 splits, tied on 6, hurt on 3
average gain: +0.010, about 0.9 penguins per test set of 81


Two findings, and both matter:

- **The split alone moves the test accuracy by about 10 points.** The same model, with the same settings, scores anywhere from about 85% to 95% depending on which penguins land in the test set. A single test score, like Module 9's 91%, is one draw from that spread.
- **Adding species gives a small gain**: about one percentage point on average, roughly one penguin per test set. That's why a single split, as in Module 9, looked like a coin flip. Across 20 splits, though, it helped far more often than it hurt (11 splits to 3, with 6 ties). That's suggestive rather than conclusive: the 20 splits reuse the same 333 penguins, so they aren't independent experiments, and a formal test would need to account for that.

This is a simple form of **repeated hold-out validation**. Cross-validation, which you'll meet in the capstone, is the same idea done more systematically. Note that each run's folder still holds its own config, metrics and log, so any one of these 40 numbers can be traced back to exactly how it was produced.

## Step 9: Document it

The last file is the one people read first. A good project README says what it does, how to install it, how to use it, and how to run the tests, with commands that can be copied.

In [27]:
%%writefile work/penguin-sex/README.md
# penguin-sex

Predict a penguin's sex from its bill, flipper and body measurements with logistic regression,
trained with Newton's method. Built on the Palmer Penguins data (CC0).

## Install

    python -m venv .venv
    source .venv/bin/activate
    pip install -e ".[dev]"

## Train

    penguin-sex train --data penguins.csv --config configs/default.json --out runs/default

Each run folder contains `model.json` (weights and preprocessing), `metrics.json`
(config, metrics, data fingerprint and library versions) and `train.log`.

## Predict

    penguin-sex predict --model runs/default/model.json --data new.csv --out predictions.csv

The input CSV needs `species` and the four measurement columns.

## Test

    pytest

Writing work/penguin-sex/README.md


In [28]:
def tree(folder, prefix=""):
    entries = sorted(p for p in folder.iterdir() if p.name not in {"runs", "__pycache__"})
    for i, path in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ("└── " if last else "├── ") + path.name + ("/" if path.is_dir() else ""))
        if path.is_dir():
            tree(path, prefix + ("    " if last else "│   "))


print("penguin-sex/")
tree(PROJECT)

penguin-sex/
├── README.md
├── configs/
│   ├── default.json
│   └── no_species.json
├── new_penguins.csv
├── penguin_sex/
│   ├── __init__.py
│   ├── cli.py
│   ├── config.py
│   ├── data.py
│   ├── model.py
│   └── persistence.py
├── predictions.csv
├── pyproject.toml
└── tests/
    ├── conftest.py
    ├── test_cli.py
    ├── test_config.py
    ├── test_data.py
    ├── test_model.py
    └── test_persistence.py


## Summary

What changed between Module 9's notebook and this project:

| Notebook version | Project version |
|---|---|
| Settings scattered through cells | One validated `TrainConfig`, loaded from JSON, unknown keys rejected |
| Checks you remember to run | 28 tests that run in under a second, including the gradient check |
| Results in cell outputs | A run folder per experiment: model, metrics, data fingerprint, versions, log |
| Re-run everything to predict | `predict` loads one file that holds weights **and** preprocessing |
| One test split | 20 splits in a loop, showing both the spread and a small, consistent gain |

And one bug that the tests caught before it could matter: a saved model without its preprocessing.

**Ideas to extend it:**

- Add `--l2` to the `train` command, and use the 20-split loop to choose a penalty.
- Add an `evaluate` command that loads a model and reports accuracy on a labelled CSV.
- Run `pip install -e ".[dev]"` in a fresh virtual environment on your own machine and check that `penguin-sex` and `pytest` work from a terminal.
- Put the project in its own git repository and add a GitHub Actions workflow that runs `pytest` on every push.